In [1]:
import dask_gateway

# Create a connection to dask-gateway.
gw = dask_gateway.Gateway("https://dask-gateway.jasmin.ac.uk", auth="jupyterhub")

# Inspect and change the options if required before creating your cluster.
options = gw.cluster_options()
options.worker_cores = 2
options.account = "nceo_generic"
# options.worker_memory = 20
# Create a Dask cluster, or, if one already exists, connect to it.
# This stage creates the scheduler job in Slurm, so it may take some
# time while your job queues.
clusters = gw.list_clusters()
if not clusters:
    cluster = gw.new_cluster(options, shutdown_on_close=False)
else:
    cluster = gw.connect(clusters[0].name)

# Create at least one worker, and allow your cluster to scale to three.
cluster.adapt(minimum=7, maximum=8)


In [2]:

# Get a Dask client.
client = cluster.get_client()
client

# print client status url
print(f"Client status url: {client.dashboard_link}")

# parent_dir = '/home/users/marcyin/marcyin/UK_crop_map'
# zarr_path = f'{parent_dir}/data/S2_31UDU_monthly_comosite_2018.zarr'


Client status url: https://dask-gateway.jasmin.ac.uk/clusters/b40e11ca057f49d1a4eb82cdaa7bd8e5/status


In [4]:
import os
import rioxarray
import xarray as xr
import numpy as np
import geopandas as gpd
import pyogrio
from tqdm import tqdm
from dask.diagnostics import ProgressBar

In [6]:

def extract_crop_samples(parent_dir, zarr_path, crop_map_file):
    """
    Extract crop samples from satellite imagery based on a crop map.
    
    Parameters:
    parent_dir (str): Path to the parent directory containing the data.
    zarr_path (str): Path to the Zarr file with the satellite imagery.
    crop_map_file (str): Path to the crop map file (FileGeoDatabase).
    
    Returns:
    geopandas.GeoDataFrame: GeoDataFrame containing the crop samples.
    """
    # Load the satellite imagery
    ds = xr.open_zarr(zarr_path)
    ds.rio.set_spatial_dims('lon', 'lat', inplace=True)

    bounds = ds.rio.transform_bounds(pyogrio.read_info(crop_map_file)['crs'])

    gdf = gpd.read_file(crop_map_file, columns=['lucode', 'geometry'], engine="pyogrio", bbox=bounds)
    gdf = gdf.to_crs(ds.rio.crs)
    
    lons = gdf.geometry.x.values
    lats = gdf.geometry.y.values
    
    # calculate the index of the nearest pixel from the geotransform
    geo_transform = ds.rio.transform().to_gdal()
    # transform the coordinates to pixel index
    x_idx = ((lons - geo_transform[0]) / geo_transform[1]).astype(int)
    y_idx = ((lats - geo_transform[3]) / geo_transform[5]).astype(int)


    # x_idx = np.clip(x_idx, 0, ds.sizes['lon'] - 1)
    # y_idx = np.clip(y_idx, 0, ds.sizes['lat'] - 1)

    # x_indx_nna = np.argmin(np.abs(lons[:1000] - ds.lon.values[:, np.newaxis]), axis=0)
    # y_indx_nna = np.argmin(np.abs(lats[:1000] - ds.lat.values[:, np.newaxis]), axis=0)
    # import pylab as plt
    # fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    # axs[0].plot(x_idx[:1000], x_indx_nna, 'o', ms=2)
    # axs[0].plot([0, x_indx_nna.max()], [0, x_indx_nna.max()], 'r--', lw=1)
    # axs[1].plot(y_idx[:1000], y_indx_nna, 'o', ms=2)
    # axs[1].plot([0, y_indx_nna.max()], [0, y_indx_nna.max()], 'r--', lw=1)


    mask = (x_idx >= 0) & (x_idx < ds.sizes['lon']) & (y_idx >= 0) & (y_idx < ds.sizes['lat'])
    x_idx = x_idx[mask]
    y_idx = y_idx[mask]
    lucode = gdf.lucode.values[mask].astype(str)

    # lucodes = ['AC00', 'AC01', 'AC03', 'AC04', 'AC05', 'AC06', 'AC07', 'AC09',
    #             'AC10', 'AC100', 'AC14', 'AC15', 'AC16', 'AC17', 'AC18', 'AC19',
    #             'AC20', 'AC22', 'AC23', 'AC24', 'AC26', 'AC27', 'AC30', 'AC32',
    #             'AC34', 'AC35', 'AC36', 'AC37', 'AC38', 'AC41', 'AC44', 'AC45',
    #             'AC50', 'AC52', 'AC58', 'AC59', 'AC60', 'AC61', 'AC62', 'AC63',
    #             'AC64', 'AC65', 'AC66', 'AC67', 'AC68', 'AC69', 'AC70', 'AC71',
    #             'AC72', 'AC74', 'AC81', 'AC88', 'AC90', 'AC92', 'AC94', 'CA02',
    #             'FA01', 'HE02', 'HEAT', 'LG01', 'LG02', 'LG03', 'LG04', 'LG06',
    #             'LG07', 'LG08', 'LG09', 'LG11', 'LG13', 'LG14', 'LG15', 'LG16',
    #             'LG20', 'LG21', 'NA01', 'NU01', 'PG01', 'SR01', 'TC01', 'WA00',
    #             'WA01', 'WO12']

    # # check if all the lucodes are in the list
    # lucode_not_in = [i for i in lucode if i not in lucodes]
    # if lucode_not_in:
    #     raise ValueError(f'Lucodes {lucode_not_in} not in the list of lucodes!')

    # lucode_index = np.array([lucodes.index(l) for l in lucode])

    band_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    with ProgressBar():
        band_vals = []
        for band_name in band_names:
            print(f'Processing {band_name}...')
            da = ds[band_name]
            da = da.where(np.isfinite(da), 0).astype('uint16')
            # now load the data into memory
            da = da.load()

            print(da.values[:, x_idx, y_idx].shape)
            # print(da.values[:, x_idx, y_idx])
            # # extract the values

            band_vals.append(da.values[:, x_idx, y_idx])

    # create one xarray dataarray for the band values with the coordindates be the band name, month(12) and the sample lucode
    band_vals = np.stack(band_vals, axis=0)
    band_vals = xr.DataArray(band_vals, dims=['band', 'month', 'sample'], 
                                        coords={'band': band_names, 
                                                 'month': np.arange(1, 13), 
                                                 'sample': np.arange(band_vals.shape[2])}
                            )
    # lucode_index = xr.DataArray(lucode_index, dims=['sample'], coords={'sample': np.arange(band_vals.shape[2])})
    
    lucode = xr.DataArray(lucode, dims=['sample'], coords={'sample': np.arange(band_vals.shape[2])})

    # create a dataset
    band_vals = xr.Dataset({'band_values': band_vals, 'lucode': lucode})
    
    print(band_vals)
    # save the dataarray
    band_vals.to_netcdf(zarr_path.replace('.zarr', '_crop_samples.nc'))
    # save the subset gdf
    gdf = gdf[mask]
    gdf.to_file(zarr_path.replace('.zarr', '_crop_samples.fgb'), driver="FlatGeobuf")
    
if __name__ == '__main__':
    year = 2021
    parent_dir = '/home/users/marcyin/marcyin/UK_crop_map'
    crop_map_file = f'{parent_dir}/CropMapOfEngland{year}-FGDB.fgb'

    zarr_path_temp = f'{parent_dir}/data/S2_*_monthly_comosite_{year}.zarr'
    from glob import glob
    zarr_paths = glob(zarr_path_temp)

    for zarr_path in zarr_paths:
        print(f'Processing {zarr_path}...')
        crop_sample_file = zarr_path.replace('.zarr', '_crop_samples.fgb')
        if not os.path.exists(crop_sample_file):
            extract_crop_samples(parent_dir, zarr_path, crop_map_file)
        else:
            print(f'{crop_sample_file} already exists!')

Processing /home/users/marcyin/marcyin/UK_crop_map/data/S2_31UDT_monthly_comosite_2021.zarr...
Processing B2...



KeyboardInterrupt



In [7]:
cluster.shutdown()